# Exploratory Data Analysis and Model Training

# 1. Model Prediction Improvement

## 1.1 Introduction

There are several methods that can help improve the prediction performance of models. Here are some commonly used techniques:
   
1. **Data Augmentation**: This refers to techniques that increase the amount of data by adding slightly modified copies of already existing data. For example, in image processing, these techniques could include rotation, scaling, flipping, etc. In text data, it can include methods like back translation or synonym replacement.


2. **Data Cleaning**: This involves taking care of missing values (by either filling them in based on existing data, or removing the data points entirely), and handling outliers (which might distort the training of the model).


3. **Feature engineering**: This is the process of creating new features from existing data that can help improve model performance. This can involve transformations of existing features, creating interaction features, or any other kind of data manipulation that creates new, useful input for the model.


4. **Model Selection**: This involves choosing the right machine learning algorithm for your specific problem. This could be a linear regression model, a decision tree, a neural network, etc. The choice depends on the nature of your data and the problem you're trying to solve.


5. **Hyperparameter tuning**: Hyperparameters are parameters that are not learned from the data but are set before the training process. c are learning rate, number of layers in a neural network, number of clusters in a K-means clustering, etc. Tuning these can often significantly improve performance. Techniques for hyperparameter tuning include grid search, random search, and more advanced methods like Bayesian optimization.


6. **Cross-validation**: This is a resampling procedure used to evaluate the performance of a model on a limited data sample. The dataset is partitioned into 'k' equally sized folds, and the model is trained on 'k-1' folds, and the remaining fold is used for testing. This process is repeated 'k' times so that we obtain a model performance score for each fold. It helps in assessing how the results of a statistical analysis will generalize to an independent data set.


7. **Regularization**: This is a technique used to prevent overfitting, which is when a model performs well on the training data but poorly on unseen data. Regularization works by adding a penalty term to the loss function that increases as the complexity of the model increases.


8. **Ensemble your model**: This refers to combining different models to improve overall performance. Techniques include Bagging (e.g., Random Forest), Boosting (e.g., Gradient Boosting, XGBoost), and Stacking.


Since we have already covered data cleaning, feature engineering in the previous sections, our attention in this section will shift to other topics, including data augmentation, model selection, ensemble model, regularization, cross-validation and hyperparameter tuning.

## 1.2 Dataset Exploration

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [ ]:
from sklearn.datasets import load_digits

# Load the digits dataset
digits = load_digits()

# Create a dataframe
# "digits.data" contains the features and "digits.target" contains the target
df = pd.DataFrame(data= np.c_[digits['data'], digits['target']],
                  columns= digits['feature_names'] + ['target'])

# Separate the features (X) and the target (y)
X = df[digits['feature_names']]
y = df['target']

# Display the dataframe
df.head()

The digit dataset contains grayscale images of hand-written digits (0 to 9) and is widely used for practicing classification algorithms in the field of machine learning. Let's break down the dataset:

**Dataset Description:**

- **Task**: Classification

- **Number of Classes**: 10 (Digits from 0 to 9)

- **Number of Samples:** 1,797

- **Number of Features:** 64 (8x8 image of each digit)

**Features:**

Each sample in the dataset is an 8x8 image of a hand-written digit. The images are represented as 64-dimensional arrays, where each element of the array corresponds to one pixel in the image. The values of the elements (pixels) range from 0 to 16, representing the grayscale intensity of the pixel. The value 0 corresponds to a white pixel, while the value 16 corresponds to a black pixel.

**Target Variable:**

The target variable (also known as labels or classes) for each sample is the digit it represents. The values of the target variable range from 0 to 9, representing the hand-written digit in the corresponding image.

In [ ]:
df.info()

## 1.3 Data Augmentation

The `augment_data` function is defined to perform data augmentation. It takes the original images and labels as input and generates augmented versions of each image. The augmentation includes adding the original image, its horizontal flip, and a 90-degree rotation. The augmented images and labels are stored in `augmented_images` and `augmented_labels`, respectively.

In [ ]:
# Load the digit dataset
digits = load_digits()
images = digits.images
labels = digits.target

The loaded dataset is unpacked into three variables: digits, images, and labels. digits is a dictionary-like object that holds the dataset, and images and labels are NumPy arrays containing the images and corresponding labels, respectively.

In [ ]:
# Data augmentation
def augment_data(images, labels):
    augmented_images = []
    augmented_labels = []
    for image, label in zip(images, labels):
        # orginal image
        augmented_images.append(image)
        augmented_labels.append(label)

        # Flipped image
        augmented_images.append(np.fliplr(image))
        augmented_labels.append(label)

        # Rotated image
        augmented_images.append(np.rot90(image, k=1))
        augmented_labels.append(label)

    return np.array(augmented_images), np.array(augmented_labels)

augmented_images, augmented_labels = augment_data(images, labels)


This code combines the original images and their augmented versions into a single dataset, resulting in `all_images` and `all_labels`.

In [ ]:
# Combine original and augmented data
all_images = np.concatenate([images, augmented_images])
all_labels = np.concatenate([labels, augmented_labels])

The `plot_images` function is defined to visualize the original images and their augmented counterparts. It uses Matplotlib to create a grid of images, with the number of rows and columns specified by `rows` and `cols`. The function displays `num_samples` samples of original and augmented images side by side for better understanding.

In [ ]:
# Visualize the original images and their augmented counterparts
def plot_images(images, labels, rows, cols):
    fig, axes = plt.subplots(rows, cols, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i], cmap='gray')
        ax.set_title(f"Label: {labels[i]}")
        ax.axis('off')
    plt.show()

num_samples = 10# Number of samples to visualize for each category
original_images_sample = images[:num_samples]
augmented_images_sample = augmented_images[:num_samples]

plot_images(original_images_sample, labels[:num_samples], 1, num_samples)
plot_images(augmented_images_sample, labels[:num_samples], 1, num_samples)

## 1.4 Data Pre-processing

The digits dataset from sklearn is a clean dataset, meaning it `doesn't have missing values`, it `doesn't contain categorical features` that need to be encoded, and it `doesn't have obvious outliers`. Therefore, some pre-processing steps like handling missing values, encoding categorical variables, or outlier treatment are not applicable in this case.

## 1.5 Feature Engineering

The digits dataset is a set of 8x8 pixel images, and each pixel in the image is a feature. There are a total of 64 features for each image. These features are already in a form that's suitable for machine learning models, so it's typically not necessary to do additional feature engineering.

### 1.5.1 Difference between Feature Engineering and Feature Selection:
The main difference between feature engineering and feature selection is their focus and purpose:

- **Feature Engineering:** Focuses on creating, transforming, or enhancing features to provide the model with more meaningful and representative information from the data.

- **Feature Selection:** Focuses on selecting a subset of relevant features to reduce model complexity, prevent overfitting, and improve model performance.

### 1.6.1 Split the data into training and test sets

# 2. Supply chain example

## 2.1 Introduction to the Dataset

The dataset at hand is a comprehensive sales record dataset provided by a a renowned consumer goods company. In here, we can it "company A". This dataset aims to facilitate the prediction of item sales quantities in each unit (EA) using various informative features. The dataset encompasses a wide range of variables that provide insights into the sales dynamics and factors influencing consumer behavior.


### Objective:

The primary objective of this dataset is to develop a predictive model that accurately estimates the sales quantity of each item (Pos_Qty_EA) based on the provided features. By leveraging the historical sales data, "company A" aims to forecast item sales more effectively, optimize inventory management, and make data-driven decisions to maximize sales revenue and profitability.

Additionally, this dataset can be utilized to gain insights into the factors that drive or hinder sales, assess the impact of promotional activities, evaluate the performance of different store banners, and analyze the influence of pricing tiers on consumer behavior.

Through detailed exploration and analysis of this dataset, Nestle can enhance its understanding of market dynamics, improve sales forecasting accuracy, and make informed business decisions that align with customer demands and preferences.

## 2.2 Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import pyplot
import seaborn as sns
import warnings

# Ignore all warnings
warnings.filterwarnings("ignore")

In [ ]:
# get all the files from github
!git clone https://github.com/MLcmore2023/MLcmore2023.git

In [ ]:
!mv ./MLcmore2023/'day2_am_morning'/* ./MLcmore2023/'day2_am_morning'/.* ./

In [ ]:
df = pd.read_csv('data.csv')

In [ ]:
df.head(2)

In [ ]:
df.info()

### Features Explanation:


1. MATERIAL: The unique identifier for each item in the dataset.
2. source: The source of the sales data (e.g., point of sale systems, online sales platform, etc.).
3. Banner: The name or identifier of the retail banner (store brand) where the item was sold.
4. Plan_Banner: The planned retail banner for the item.
5. Plan_Region: The planned region for the item's sales.
6. CL4Key: The identifier for a higher-level category or classification level 4 of the item.
7. CL6Key: The identifier for a lower-level category or classification level 6 of the item.
8. Pos_Qty_EA: The sales quantity of the item in each unit (target variable).
9. Pos_Sales: The sales amount or revenue generated from the item.
10. POS_QTY_CS: The sales quantity of the item in case units (CS stands for case).
11. UBP: The unit buying price, which represents the cost of purchasing the item.
12. UNIT: The unit of measure for the item's sales quantity (EA or CS).
13. FACTOR_EACH: The conversion factor between each unit (EA) and case unit (CS).
14. PER_SALES_UOM_CASE: The sales quantity per unit of measure (case) for the item.
15. Complete_PPG: The identifier for a complete product group, which represents a broader category or grouping of items.
16. Total_Sales: The total sales amount or revenue for all items.
17. Baseline_Qty: The baseline or expected sales quantity for the item.
18. Baseline_Nps: The baseline or expected net promoter score (NPS) associated with the item.
19. Incr_Sales: The incremental or additional sales generated by a promotional activity or event.
20. PromoId: The identifier for a specific promotional activity or event.
21. PromoDuration: The duration of the promotional activity or event.
22. MATERIALDESC: A description or name of the item.
23. SUB_CATEGORY: The identifier for a sub-category of the item.
24. SUB_CATEGORY_DESC: A description or name of the sub-category.
25. CATEGORY: The identifier for the category of the item.
26. CATEGORY_DESC: A description or name of the category.
27. FGroup_Desc: A description or name of the item's functional group.


## 2.3 Data Cleaning

### 2.3.1 Check for Missing Values

In [ ]:
# Check for missing values
print(df.isnull().sum())

Since the datasets do not contain any missing values, we can proceed to the next step. Let us now examine each feature one by one.

### 2.3.2 Check for Duplicate Values

In [ ]:
duplicate_rows = df[df.duplicated(keep=False)]
if not duplicate_rows.empty:
    print("Duplicate rows found:")
    print(duplicate_rows)
else:
    print("No duplicate rows found in the dataset.")

Duplicate entries have been detected. We'll proceed to remove them.

In [ ]:
# Drop duplicate rows
df = df.drop_duplicates()

In [ ]:
duplicate_rows = df[df.duplicated(keep=False)]
if not duplicate_rows.empty:
    print("Duplicate rows found:")
    print(duplicate_rows)
else:
    print("No duplicate rows found in the dataset.")

## 2.4 Data Encoding

### 2.4.1 Find all the Categorical Data (excluding integer and float columns)

In [ ]:
categorical_columns = df.select_dtypes(exclude=['int', 'float']).columns

print("Categorical columns:")
print(categorical_columns)
print("There are " + str(len(categorical_columns)) + " categorical features.")

### 2.4.2 Check the Number of Unique Values in columns with Categorical Data

In [ ]:
# create a dataframe with all the categorical data from the original dataset
df_categorical = df[categorical_columns]
df_categorical.head(2)

In [ ]:
unique_value_counts = df_categorical.nunique()

print("Number of unique values in each column:")
print(unique_value_counts)

Based on the provided information, which includes the categorical dataset (`df_categorical`) and the number of unique values in each column, it is recommended to separate the following columns from the rest: "SRC_BAN_POS", "pos_date", "ForecastDate", "InStoreStart", and "InStoreEnd". These columns represent time.

### 2.4.3 Function that Performs Label Encoding on Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

def label_encode_features(df, features):
    """
    Perform label encoding on specified features in a DataFrame.

    Label encoding involves converting each unique string value into a unique integer.
    For instance, for a column ['cat', 'dog', 'dog', 'bird'], after label encoding,
    this might become [0, 1, 1, 2] depending on the ordering of unique values detected by the encoder.

    Args:
    - df (pandas.DataFrame): The input DataFrame that contains features to be encoded.
    - features (list): A list of column names (strings) in df that need to be label encoded.

    Returns:
    - pandas.DataFrame: A DataFrame with the specified features label encoded.
    """

    # Create a copy of the original DataFrame to avoid modifying it directly
    encoded_df = df.copy()

    # Initialize the LabelEncoder
    encoder = LabelEncoder()

    # Loop through each feature in the provided list
    for feature in features:
        # Fit the encoder on the feature and transform it.
        # The transformed feature is then overwritten in the DataFrame.
        encoded_df[feature] = encoder.fit_transform(encoded_df[feature])

    # Return the DataFrame with encoded features
    return encoded_df


In [ ]:
df = label_encode_features(df, categorical_columns)

In [ ]:
df.head(2)

Up to this point, we have successfully encoded all categorical features.

In [ ]:
df.info()

## 2.5 Data Visualization

### 2.5.1 Distribution of Pos_Qty_EA (target value)

In [ ]:
# Distribution plot of the target variable
plt.figure(figsize=(8, 6))
sns.histplot(data=df, x='Pos_Qty_EA', kde=True)
plt.xlabel('Pos_Qty_EA')
plt.ylabel('Frequency')
plt.title('Distribution of Pos_Qty_EA')
plt.show()

The distribution plot of the target variable Pos_Qty_EA showcases a majority of values concentrated between 0 and 500, with a notable peak in frequency within this range.

### 2.5.2 Relationship between Pos_Qty_EA and Total_Sales

In [ ]:
# Scatter plot
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Pos_Qty_EA', y='Total_Sales')
plt.xlabel('Pos_Qty_EA')
plt.ylabel('Total_Sales')
plt.title('Scatter Plot: Pos_Qty_EA vs Total_Sales')
plt.show()

The scatter plot demonstrates a positive correlation between Pos_Qty_EA and Total_Sales, highlighting a potential pattern indicating that higher values of Pos_Qty_EA tend to correspond with higher values of Total_Sales.

## 2.6 Feature Engineering

### 2.6.1 Correlation Heapmap

In [ ]:
# Correlation heatmap
plt.figure(figsize=(25, 20))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', square=True)
plt.title('Correlation Heatmap')
plt.show()


Representing the correlation between numerical features in this dataset, the color-coded heatmap provides insights into the relationships and dependencies among different features, including the target variable 'Pos_Qty_EA'. It highlights the strength and direction of these correlations.

### 2.6.2 Calculation and Ranking of Feature Correlation with the Target Variable

In [ ]:
# Calculate the correlation between each feature and the target variable
correlation = df.corr()['Pos_Qty_EA'].drop('Pos_Qty_EA')

# Sort the correlation values in descending order
correlation = correlation.abs().sort_values(ascending=False)

# Print the feature correlation values
print(correlation)

### 2.6.3 Identification of Highly Correlated Features with the Target Variable based on a Threshold

In [ ]:
# Set the correlation threshold
threshold = 0.5

# Filter features based on the correlation threshold
highly_correlated_features = correlation[correlation.abs() >= threshold]

# Print the highly correlated features
print(highly_correlated_features)

In [ ]:
# These are the selected features
threshold_selected_features = highly_correlated_features.index.tolist()
threshold_selected_features

### 2.6.4 Feature selection using Recursive Feature Elimination (RFE)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

# Split the dataset into X (features) and y (target variable)
X = df.drop(columns=["Pos_Qty_EA"])
y = df["Pos_Qty_EA"]

# Initialize a random forest regressor
rf = RandomForestRegressor(n_estimators=100, max_features="sqrt")

# Use RFE for feature selection
top_k = 5  # Number of top features to select
rfe = RFE(estimator=rf, n_features_to_select=top_k, step=1)
rfe = rfe.fit(X, y)

# Get the selected features
rfe_selected_features = X.columns[rfe.support_].tolist()

In [ ]:
rfe_selected_features =list(rfe_selected_features)
rfe_selected_features

## 2.7 Model Selection

In this section, we will train multiple models using the dataset and evaluate their performance to select the model that achieves the highest level of accuracy. By comparing the accuracy of different models, we can determine which one is the most effective for this particular task. This process will enable us to make an informed decision about the best model to use for prediction or further analysis.


### 2.7.1 Import Libraries

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_squared_error, accuracy_score

### 2.7.2 Model Comparison

In [ ]:
# Separate features and target
X = df.drop('Pos_Qty_EA', axis=1)  # Features
y = df['Pos_Qty_EA']  # Target

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define a dictionary to store model names and instances
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(),
    'Random Forest': RandomForestRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(),
    'Support Vector Machine': SVR(),
    'Neural Network': MLPRegressor(),
    'K-Nearest Neighbors': KNeighborsRegressor(),
    'Naive Bayes': GaussianNB()
}

# Create an empty list to store accuracy scores
accuracy_list = []

# Iterate over the models, train, and evaluate
for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred.round())  # Rounding the predictions for classification
    accuracy_list.append({'Model': model_name, 'Accuracy': accuracy})

# Create a DataFrame from the list
accuracy_df = pd.DataFrame(accuracy_list)

In [ ]:
# rank the model based on the accuracy
accuracy_df = accuracy_df.sort_values('Accuracy', ascending=False)
accuracy_df

Therefore, the decision forest model and random forest has been chosen as the candidate models based on its performance and accuracy during the evaluation process.